# <span style="background-color: #fffcb2; color: black">**핑핑이 2차 과제**</span>

In [1]:
!pip install requests beautifulsoup4

In [ ]:
"""
공차 영양성분정보 크롤러 (정적 크롤링: requests + BeautifulSoup)

대상 페이지: https://www.gong-cha.co.kr/brand/menu/product_nutrition
- 이 페이지는 화면에 보이는 메뉴 목록과 달리, 영양성분 전체를 표(table)로 모아 보여주는
  별도의 정적 페이지입니다. 최초 HTML에 이미 모든 메뉴의 영양정보가 텍스트로 들어 있습니다.

[반복 단위 / 셀렉터]
- 페이지 안에 카테고리별로 여러 개의 <table> 이 있는 구조로 추정됩니다
  (NEW 시즌, 커피, 티 등 섹션별 표가 이어짐).
- 표 하나 안에서 헤더 행(thead의 tr, 또는 없으면 첫 번째 tr)의 th/td 글자를 읽어
  "메뉴명", "사이즈", "구분", "열량", "당류", "포화지방", "카페인" 같은 열 이름이
  몇 번째 칸(인덱스)에 있는지 먼저 파악합니다.
- 그다음 나머지 행(tr)마다 그 인덱스에 해당하는 칸(td)의 글자를 꺼내
  헤더 이름 -> 값 형태의 딕셔너리로 만듭니다.
- class 이름에 의존하지 않고 "헤더 글자가 어떤 키워드를 포함하는지"로 열을 찾기 때문에,
  표마다 class 구성이 달라도 비교적 안정적으로 동작합니다.

[필터 조건]
- 사이즈: "L" (정확히 L인 경우만; J, G 등은 제외)
- 구분(온도): "COLD" (HOT은 제외)

[결과 열]
- 메뉴명, 열량(kcal), 당류(g), 포화지방(g), 카페인(mg)  (5열, 단위 포함)

[처리 규칙]
- 헤더에서 필요한 열을 찾지 못하는 표는 건너뜁니다.
- 값이 없는 칸은 빈 문자열로 저장합니다.
- 요청에는 timeout을 지정하고, HTTP 오류는 raise_for_status()로 확인합니다.
"""


HTTP 상태 코드: 200
조건(사이즈=L, 구분=COLD) 일치 전체 건수: 112건
저장된 건수(상위 20건): 20건
gongcha_nutrition_L_cold.csv 저장 완료


In [16]:
import csv

import requests
from bs4 import BeautifulSoup

url = "https://www.gong-cha.co.kr/brand/menu/product_nutrition"

# 표(table)의 열 순서 (헤더 기준):
# 0 메뉴명 | 1 구분(HOT/COLD) | 2 사이즈(L/J/G) | 3 컵 용량 |
# 4 열량 | 5 당류 | 6 단백질 | 7 포화지방 | 8 나트륨 | 9 카페인

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    rows = []
    for table in soup.select("table"):
        for tr in table.select("tr"):
            cells = tr.find_all("td")  # 헤더 행은 th라서 자동으로 제외됨
            if len(cells) < 10:
                continue

            name = cells[0].get_text(strip=True)
            temp = cells[1].get_text(strip=True).upper()
            size = cells[2].get_text(strip=True).upper()

            if size != "L" or temp != "COLD":
                continue

            rows.append(
                [
                    name,
                    cells[4].get_text(strip=True),  # 열량
                    cells[5].get_text(strip=True),  # 당류
                    cells[7].get_text(strip=True),  # 포화지방
                    cells[9].get_text(strip=True),  # 카페인
                ]
            )

    rows = rows[:20]  # 20건만 저장

    with open("gongcha_nutrition_L_cold.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["메뉴명", "열량(kcal)", "당류(g)", "포화지방(g)", "카페인(mg)"])
        writer.writerows(rows)

    print(f"HTTP 상태 코드: {response.status_code}")
    print(f"수집 건수: {len(rows)}건")
    print("gongcha_nutrition_L_cold.csv 저장 완료")

except requests.RequestException as e:
    print(f"HTTP 요청 오류: {e}")

HTTP 상태 코드: 200
수집 건수: 20건
gongcha_nutrition_L_cold.csv 저장 완료


In [17]:
import pandas as pd

df = pd.read_csv('gongcha_nutrition_L_cold.csv')
df.index = range(1, len(df) + 1)   # 인덱스 1~20
df.index.name = '번호'             # (선택) 인덱스 이름

df                                            # print(df) 대신 df로 출력하면 표 형태로 보입니다


,메뉴명,열량(kcal),당류(g),포화지방(g),카페인(mg)
번호,,,,,
1,초콜릿 시그니처 밀크티+펄,540,31,17,43
2,초코바른 딸기 스무디 with 쿠키,465,44,16,43
3,참외 자스민티,240,55,0,40
4,핑크구아바 밀크티,237,47,2,60
5,핑크구아바 봉봉 아이스티,144,35,0,40
6,포도 말랑 밀크티,412,47,12,175
7,포도 말랑 크러쉬,261,57,0,32
8,망고 트로피컬 블러쉬 스무디,346,69,0,25
9,초당 옥슈슈 밀크티,461,61,5,28
